# C4a · Timbre: the shape of the spectrum

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juansgomez87/py-mir-toolbox/blob/main/docs/chapters/C4a_spectral_shape.ipynb)

*Timbre* is what makes a flute and a trumpet sound different when they play the same note at the same level. Much of it is in the **spectrum**: which frequencies the sound contains, and how strong each one is.

This chapter describes the shape of the spectrum with a few numbers from MIRtoolbox:

- the **zero-crossing rate**, a quick measure of noisiness taken from the waveform;
- the **spectral centroid, spread, skewness and kurtosis**, which describe where the energy sits and how it is spread;
- the **spectral flatness and entropy**, which tell a clear tone from noise;
- the **roll-off and brightness**, which measure the amount of high frequencies.

Each section has the same steps as in C3: ① the idea, with sound; ② the maths in NumPy; ③ mirpy; ④ librosa; ⑤ Essentia; ⑥ the three libraries side by side; and an exercise.

## Setup

In Colab, the next cell installs the libraries. On your own computer, see *Getting started*.

In [ ]:
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic(
        "pip", "install -q git+https://github.com/juansgomez87/py-mir-toolbox librosa essentia"
    )

In [ ]:
import librosa
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio
from scipy.signal import sawtooth

import mirpy as mir

try:
    import essentia.standard as es
except ImportError:
    es = None
    print("Essentia is not installed here; the Essentia cells will be skipped.")

Three synthetic sounds of one second each, with very different timbres:

- a **sine** tone at 440 Hz: a single frequency, a soft and "pure" sound;
- a **sawtooth** wave at 440 Hz: the same note, with many harmonics, a buzzy and bright sound;
- white **noise**: all frequencies at once, with no pitch.

We join them into one three-second sound, so that each feature curve below has three parts.

In [ ]:
sr = 22050
t = np.arange(sr) / sr  # one second
rng = np.random.default_rng(0)

sine = 0.5 * np.sin(2 * np.pi * 440 * t)
saw = 0.5 * sawtooth(2 * np.pi * 440 * t)
noise = rng.uniform(-0.5, 0.5, t.size)  # white noise, never beyond ±0.5
sounds = {"sine": sine, "sawtooth": saw, "noise": noise}

three_parts = np.concatenate([sine, saw, noise])
Audio(three_parts, rate=sr, normalize=False)

All features use MIRtoolbox's frames from C3: 50 ms long, with a hop of half a frame. We give librosa and Essentia the same frame length and hop.

In [ ]:
frame_length = int(0.05 * sr)  # 1102 samples
hop = int(0.025 * sr)  # 551 samples (librosa and Essentia need a whole number)
n_fft = 2048  # FFT size: the next power of two above the frame length, as in MIRtoolbox


def essentia_times(n_frames):
    """Centre time of each Essentia frame (first frame starting at 0)."""
    return (np.arange(n_frames) * hop + frame_length / 2) / sr


def part_lines(ax):
    """Mark the borders between the sine, sawtooth and noise parts."""
    for border in (1, 2):
        ax.axvline(border, color="lightgray", linewidth=1)

## 1. The spectrum, in brief

### ① Idea

Any sound can be described as a sum of sine waves of different frequencies. The **spectrum** shows how strong each frequency is in a frame. A sine has one peak; a sawtooth has a peak at the note and at every multiple of it (the **harmonics**); noise has energy everywhere.

### ② From scratch

Three steps for one frame:

1. multiply the frame by a **window**, a bell-shaped curve that fades the edges to zero (MIRtoolbox uses a Hamming window);
2. compute the **FFT** (fast Fourier transform), which measures every frequency at once;
3. keep the **magnitude** (the size) of each frequency.

In [ ]:
def spectrum_from_scratch(frame):
    """Magnitude spectrum of one frame: Hamming window, then FFT."""
    windowed = frame * np.hamming(frame.size)
    return np.abs(np.fft.rfft(windowed, n=n_fft))


freqs = np.fft.rfftfreq(n_fft, d=1 / sr)  # frequency of each FFT bin, in Hz

fig, axes = plt.subplots(1, 3, figsize=(11, 3), sharey=True)
for ax, (name, x) in zip(axes, sounds.items(), strict=True):
    ax.plot(freqs, spectrum_from_scratch(x[:frame_length]))
    ax.set_title(name)
    ax.set_xlabel("Frequency (Hz)")
    ax.set_xlim(0, 6000)
axes[0].set_ylabel("Magnitude")
plt.show()

### ③ mirpy

`mir.spectrum` is MIRtoolbox `mirspectrum`. With `frame=True` you get one spectrum per frame. The spectral features below call it for you when you give them a sound.

In [ ]:
a = mir.audio(three_parts, sr=sr)
spec = mir.spectrum(a, frame=True)
print(spec.data.shape)  # (channels, frequencies, frames)

frame_index = 60  # a frame in the sawtooth part
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(spec.x_values, spec.data[0, :, frame_index])
ax.set_xlim(0, 6000)
ax.set_xlabel("Frequency (Hz)")
ax.set_title(f"mirpy spectrum of the frame at {spec.frame_centers[frame_index]:.2f} s")
plt.show()

Over all frames, the spectra make a **spectrogram**: time on the horizontal axis, frequency on the vertical axis, and colour for the magnitude (here in decibels, so that weak harmonics are visible).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.pcolormesh(
    spec.frame_centers, spec.x_values, 20 * np.log10(spec.data[0] + 1e-6), shading="auto"
)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Frequency (Hz)")
ax.set_title("sine | sawtooth | noise")
plt.show()

### ④ librosa

`librosa.stft` (short-time Fourier transform) computes the spectra of all frames at once. By default it uses a Hann window, which is very close to Hamming, and centres the frames as in C3. Here we ask for a Hamming window and MIRtoolbox's frame length.

In [ ]:
librosa_spec = np.abs(
    librosa.stft(
        three_parts, n_fft=n_fft, hop_length=hop, win_length=frame_length, window="hamming"
    )
)
librosa_freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
print(librosa_spec.shape)  # (frequencies, frames)

### ⑤ Essentia

In Essentia, each step is its own algorithm: `FrameGenerator` cuts the frames, `Windowing` applies the window (and pads the frame with zeros up to the FFT size), and `Spectrum` gives the magnitudes. `Windowing` rescales the window by default; `normalized=False` keeps the same scale as the other libraries.

In [ ]:
def essentia_spectra(x):
    """Magnitude spectrum of each frame with Essentia, one row per frame."""
    window = es.Windowing(
        type="hamming", size=frame_length, zeroPadding=n_fft - frame_length, normalized=False
    )
    spectrum = es.Spectrum(size=n_fft)
    frames = es.FrameGenerator(
        x.astype(np.float32), frameSize=frame_length, hopSize=hop, startFromZero=True
    )
    return np.array([spectrum(window(frame)) for frame in frames])


if es is None:
    print("Essentia is not available; skipped.")
else:
    essentia_spec = essentia_spectra(three_parts)
    print(essentia_spec.shape)  # (frames, frequencies)

### ⑥ Side by side

The spectrum of one sawtooth frame from each library. The peaks fall at the same frequencies: 440 Hz and its multiples.

In [ ]:
librosa_index = np.argmin(np.abs(librosa.times_like(librosa_spec, sr=sr, hop_length=hop) - 1.5))

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(spec.x_values, spec.data[0, :, frame_index], linewidth=3, alpha=0.5, label="mirpy")
ax.plot(librosa_freqs, librosa_spec[:, librosa_index], label="librosa")
if es is not None:
    ax.plot(librosa_freqs, essentia_spec[frame_index], linestyle="--", label="Essentia")
ax.set_xlim(0, 3000)
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Magnitude")
ax.legend()
plt.show()

- **mirpy** and **Essentia** work frame by frame, starting at time 0.
- **librosa** returns the whole spectrogram as one array, with centred frames.
- All three give the magnitude of each frequency without rescaling it, so the heights match too.

### Exercise

Make a chord of three sine tones: A4 (440 Hz), C♯5 (554 Hz) and E5 (659 Hz). Listen to it, plot the spectrum of one frame with `mir.spectrum`, and find the three peaks.

In [ ]:
# Your answer here.

Solution (click to show):

In [ ]:
chord = sum(0.3 * np.sin(2 * np.pi * f * t) for f in (440, 554, 659))
chord_spec = mir.spectrum(mir.audio(chord, sr=sr), frame=True)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(chord_spec.x_values, chord_spec.data[0, :, 10])
ax.set_xlim(0, 1000)
ax.set_xlabel("Frequency (Hz)")
plt.show()

Audio(chord, rate=sr)

## 2. Zero-crossing rate

### ① Idea

The zero-crossing rate counts how often the waveform changes sign, from negative to positive or back. MIRtoolbox calls it "a simple indicator of noisiness". A low tone crosses zero slowly; noise jumps up and down all the time.

This feature works on the waveform, not the spectrum. It is in this chapter because it describes timbre.

In [ ]:
shown = slice(0, 200)  # the first 200 samples, about 9 ms
fig, axes = plt.subplots(1, 3, figsize=(11, 2.5), sharey=True)
for ax, (name, x) in zip(axes, sounds.items(), strict=True):
    ax.plot(t[shown] * 1000, x[shown])
    ax.axhline(0, color="gray", linewidth=0.8)
    ax.set_title(name)
    ax.set_xlabel("Time (ms)")
plt.show()

### ② From scratch

A sign change happens between two samples when their product is negative. MIRtoolbox counts the changes in **one direction** only (that is, half of all changes), divides by the number of samples, and multiplies by the sampling rate to get changes **per second**. For a sine, that is about its frequency.

In [ ]:
def zero_crossing_rate(frame):
    """Sign changes in one direction per second, as in MIRtoolbox."""
    changes = np.sum(frame[1:] * frame[:-1] < 0)
    return changes / 2 / frame.size * sr


for name, x in sounds.items():
    print(f"{name:>8}: {zero_crossing_rate(x[:frame_length]):6.0f} per second")

### ③ mirpy

`mir.zerocross` is MIRtoolbox `mirzerocross`. Two options change the unit: `per="sample"` (MATLAB `'Per', 'Sample'`) and `direction="both"` (MATLAB `'Dir', 'Both'`).

In [ ]:
zc = mir.zerocross(a, frame=True)
zc.plot()
plt.show()

### ④ librosa

`librosa.feature.zero_crossing_rate` counts the changes in **both** directions, per **sample**. To compare with MIRtoolbox, divide by 2 and multiply by the sampling rate.

In [ ]:
librosa_zc = librosa.feature.zero_crossing_rate(
    three_parts, frame_length=frame_length, hop_length=hop
)[0]
librosa_zc_times = librosa.times_like(librosa_zc, sr=sr, hop_length=hop)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(librosa_zc_times, librosa_zc)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Changes per sample")
ax.set_title("librosa.feature.zero_crossing_rate")
plt.show()

### ⑤ Essentia

Essentia's `ZeroCrossingRate` also counts both directions per sample.

In [ ]:
if es is None:
    print("Essentia is not available; skipped.")
else:
    zcr_algorithm = es.ZeroCrossingRate()
    essentia_zc = np.array(
        [
            zcr_algorithm(frame)
            for frame in es.FrameGenerator(
                three_parts.astype(np.float32),
                frameSize=frame_length,
                hopSize=hop,
                startFromZero=True,
            )
        ]
    )

### ⑥ Side by side

The three curves, all converted to MIRtoolbox's unit (changes in one direction per second).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(zc.frame_centers, zc.data[0, 0], linewidth=3, alpha=0.5, label="mirpy")
ax.plot(librosa_zc_times, librosa_zc / 2 * sr, label="librosa ÷ 2 × sr")
if es is not None:
    ax.plot(
        essentia_times(essentia_zc.size), essentia_zc / 2 * sr, "--", label="Essentia ÷ 2 × sr"
    )
part_lines(ax)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Changes per second")
ax.legend()
plt.show()

- **mirpy** follows MIRtoolbox: one direction, per second.
- **librosa** and **Essentia** count both directions, per sample. The curves have the same shape; only the unit differs.
- The sine and the sawtooth have the same rate (440 per second): the zero-crossing rate hears the pitch, not the harmonics.

### Exercise

Compute the zero-crossing rate with `mir.zerocross` for sine tones of 100, 200, 400 and 800 Hz. What do you notice? What happens if you add a little noise to the 100 Hz tone?

In [ ]:
# Your answer here.

Solution (click to show):

In [ ]:
for f0 in (100, 200, 400, 800):
    tone = mir.audio(np.sin(2 * np.pi * f0 * t), sr=sr)
    print(f0, "Hz:", round(mir.zerocross(tone).data.item()))

noisy = np.sin(2 * np.pi * 100 * t) + 0.05 * rng.standard_normal(t.size)
print("100 Hz + noise:", round(mir.zerocross(mir.audio(noisy, sr=sr)).data.item()))

For a sine, the rate equals the frequency. A little noise adds many extra crossings near zero, so the rate jumps: the zero-crossing rate is very sensitive to noise.

## 3. Spectral centroid, spread, skewness and kurtosis

### ① Idea

Think of the spectrum as a pile of sand along the frequency axis. Four numbers describe its shape:

- the **centroid** is its centre of mass: the "average frequency". A high centroid sounds bright;
- the **spread** is how wide the pile is around the centroid (its standard deviation);
- the **skewness** tells whether the pile leans: a positive value means a long tail towards high frequencies;
- the **kurtosis** tells how peaked the pile is: a high value means a narrow peak with little around it.

Statisticians call these the first four **moments** of a distribution.

### ② From scratch

Divide the spectrum by its sum, so that it adds up to 1 like a set of probabilities $p_k$. Then, with $f_k$ the frequency of bin $k$:

$$\text{centroid } \mu = \sum_k f_k\, p_k \qquad \text{spread } \sigma = \sqrt{\sum_k (f_k - \mu)^2\, p_k}$$

$$\text{skewness} = \frac{\sum_k (f_k - \mu)^3\, p_k}{\sigma^3} \qquad \text{kurtosis} = \frac{\sum_k (f_k - \mu)^4\, p_k}{\sigma^4}$$

In [ ]:
def spectral_moments(magnitudes):
    """Centroid, spread, skewness and kurtosis of one magnitude spectrum."""
    p = magnitudes / magnitudes.sum()
    centroid = np.sum(freqs * p)
    spread = np.sqrt(np.sum((freqs - centroid) ** 2 * p))
    skewness = np.sum((freqs - centroid) ** 3 * p) / spread**3
    kurtosis = np.sum((freqs - centroid) ** 4 * p) / spread**4
    return centroid, spread, skewness, kurtosis


fig, axes = plt.subplots(1, 3, figsize=(11, 3), sharey=True)
for ax, (name, x) in zip(axes, sounds.items(), strict=True):
    magnitudes = spectrum_from_scratch(x[:frame_length])
    centroid, spread, skewness, kurtosis = spectral_moments(magnitudes)
    ax.plot(freqs, magnitudes, color="gray")
    ax.axvline(centroid, color="C3", label="centroid")
    ax.axvspan(centroid - spread, centroid + spread, color="C3", alpha=0.15, label="± spread")
    ax.set_title(f"{name}\nskewness {skewness:.1f}, kurtosis {kurtosis:.1f}")
    ax.set_xlabel("Frequency (Hz)")
axes[0].legend()
plt.show()

The sine has a low centroid, a narrow spread, and a very high skewness and kurtosis: one sharp peak, with a long empty tail above it. Noise has its centroid in the middle, a wide spread, and no lean (skewness near 0).

### ③ mirpy

`mir.centroid`, `mir.spread`, `mir.skewness` and `mir.kurtosis` are the MIRtoolbox functions of the same names. Given a sound, they compute its spectrum first.

In [ ]:
moments = {
    "centroid (Hz)": mir.centroid(a, frame=True),
    "spread (Hz)": mir.spread(a, frame=True),
    "skewness": mir.skewness(a, frame=True),
    "kurtosis": mir.kurtosis(a, frame=True),
}

fig, axes = plt.subplots(4, 1, figsize=(8, 7), sharex=True)
for ax, (name, curve) in zip(axes, moments.items(), strict=True):
    ax.plot(curve.frame_centers, curve.data[0, 0])
    part_lines(ax)
    ax.set_ylabel(name)
axes[3].set_yscale("log")
axes[-1].set_xlabel("Time (s)")
plt.show()

The centroid curve has a gap in the noise part. MIRtoolbox gives **no value** (NaN, "not a number") for frames whose spectrum is almost flat, because a centre of mass means little there. The manual calls this option `'MaxEntropy'`; in mirpy it is `max_entropy`. Frames that are almost silent get no value either (`'MinRMS'`, `min_rms`). To get a value everywhere, turn the first check off:

In [ ]:
print(np.isnan(moments["centroid (Hz)"].data).sum(), "frames without a value")
print(np.isnan(mir.centroid(a, frame=True, max_entropy=None).data).sum(), "with max_entropy=None")

Two more MIRtoolbox details, which mirpy keeps so that the results match (see the [Parity report](../parity.md)): before the moments are computed, the spectrum's smallest value is added to it; and the kurtosis is *not* reduced by 3, although the manual describes it that way. A bell-shaped distribution has a kurtosis of 3 here.

### ④ librosa

librosa has `spectral_centroid` and `spectral_bandwidth`. The bandwidth with its default `p=2` is the spread. There is no skewness or kurtosis; you can compute them from `librosa.stft` with the formulas in ②.

In [ ]:
librosa_centroid = librosa.feature.spectral_centroid(S=librosa_spec, sr=sr, n_fft=n_fft)[0]
librosa_spread = librosa.feature.spectral_bandwidth(S=librosa_spec, sr=sr, n_fft=n_fft)[0]
librosa_times = librosa.times_like(librosa_centroid, sr=sr, hop_length=hop)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(librosa_times, librosa_centroid, label="centroid")
ax.plot(librosa_times, librosa_spread, label="spread (bandwidth)")
part_lines(ax)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Hz")
ax.legend()
plt.show()

### ⑤ Essentia

Essentia's `Centroid` works on positions from 0 to `range`; set `range` to half the sampling rate to get Hz. `CentralMoments` followed by `DistributionShape` gives the spread (as a variance, so take its square root), skewness and kurtosis. Essentia's kurtosis *is* reduced by 3.

In [ ]:
def essentia_moments(spectra):
    """Centroid, spread, skewness and kurtosis of each Essentia spectrum."""
    centroid = es.Centroid(range=sr / 2)
    central_moments = es.CentralMoments(range=sr / 2)
    shape = es.DistributionShape()
    rows = []
    for magnitudes in spectra:
        variance, skewness, kurtosis = shape(central_moments(magnitudes))
        rows.append((centroid(magnitudes), np.sqrt(variance), skewness, kurtosis))
    return np.array(rows).T


if es is None:
    print("Essentia is not available; skipped.")
else:
    ess_centroid, ess_spread, ess_skewness, ess_kurtosis = essentia_moments(essentia_spec)
    ess_times = essentia_times(ess_centroid.size)

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(ess_times, ess_centroid, label="centroid")
    ax.plot(ess_times, ess_spread, label="spread")
    part_lines(ax)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Hz")
    ax.legend()
    plt.show()

### ⑥ Side by side

The centroid from the three libraries:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
centroid_curve = moments["centroid (Hz)"]
ax.plot(
    centroid_curve.frame_centers, centroid_curve.data[0, 0], linewidth=3, alpha=0.5, label="mirpy"
)
ax.plot(librosa_times, librosa_centroid, label="librosa")
if es is not None:
    ax.plot(ess_times, ess_centroid, "--", label="Essentia")
part_lines(ax)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Centroid (Hz)")
ax.legend()
plt.show()

- The three curves follow the same steps: low for the sine, high for the sawtooth and the noise.
- **mirpy** leaves out the noise frames (flat spectrum) and silent frames; **librosa** and **Essentia** always give a value.
- For skewness and kurtosis, only **mirpy** and **Essentia** have functions, and Essentia's kurtosis is 3 lower.
- In the sine part, the librosa and Essentia curves drift slowly upward. Their hop is a whole number of samples, so each frame catches the wave at a slightly different point, which changes the small spill of energy around the peak. The centroid of a pure tone is sensitive to such small details.

### A real recording

A jazz trumpet phrase by Mihai Sorohan ("solo trumpet 06", [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/), from [Freesound](https://freesound.org/s/77711/); librosa downloads it the first time). Listen for how the tone gets brighter on the louder notes.

In [ ]:
trumpet = mir.audio(librosa.ex("trumpet"))
Audio(trumpet.data[0, :, 0], rate=trumpet.sr)

In [ ]:
trumpet_centroid = mir.centroid(trumpet, frame=True)
trumpet_rms = mir.rms(trumpet, frame=True)

fig, axes = plt.subplots(2, 1, figsize=(9, 4), sharex=True)
axes[0].plot(trumpet_rms.frame_centers, trumpet_rms.data[0, 0])
axes[0].set_ylabel("RMS")
axes[1].plot(trumpet_centroid.frame_centers, trumpet_centroid.data[0, 0])
axes[1].set_ylabel("Centroid (Hz)")
axes[1].set_xlabel("Time (s)")
plt.show()

The gaps in the centroid are the pauses between notes, where the sound is almost silent.

### Exercise

Compute the average centroid (`np.nanmean`) of sawtooth waves at 110, 220, 440 and 880 Hz. Plot it against the pitch. Does a higher note always sound brighter?

In [ ]:
# Your answer here.

Solution (click to show):

In [ ]:
pitches = [110, 220, 440, 880]
centroids = [
    np.nanmean(mir.centroid(mir.audio(0.5 * sawtooth(2 * np.pi * f0 * t), sr=sr), frame=True).data)
    for f0 in pitches
]

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(pitches, centroids, marker="o")
ax.set_xscale("log")
ax.set_xticks(pitches, [str(p) for p in pitches])
ax.minorticks_off()
ax.set_xlabel("Pitch (Hz)")
ax.set_ylabel("Average centroid (Hz)")
plt.show()

The centroid rises with the pitch, because all the harmonics move up with the note. Brightness and pitch are linked, so compare the centroids of different instruments on the same note.

## 4. Spectral flatness and entropy

### ① Idea

Both features tell a clear tone from noise:

- a tone has a few sharp peaks: low flatness, low entropy;
- noise spreads its energy over all frequencies: flatness and entropy close to 1.

### ② From scratch

The **flatness** divides the geometric mean of the spectrum by its ordinary (arithmetic) mean. The geometric mean is pulled down by small values, so it is much smaller than the ordinary mean when the spectrum has a few peaks and many low values.

The **entropy** (Shannon, 1948) comes from information theory. MIRtoolbox scales the spectrum to sum to 1, computes $-\sum_k p_k \log p_k$, and divides by $\log N$ (the entropy of a perfectly flat spectrum with $N$ bins), so that the result is between 0 and 1.

In [ ]:
def flatness_from_scratch(magnitudes):
    """Geometric mean divided by arithmetic mean."""
    geometric_mean = np.exp(np.mean(np.log(magnitudes)))
    return geometric_mean / np.mean(magnitudes)


def entropy_from_scratch(magnitudes):
    """Shannon entropy of the normalised spectrum, divided by its largest possible value."""
    p = magnitudes / magnitudes.sum()
    return -np.sum(p * np.log(p)) / np.log(p.size)


for name, x in sounds.items():
    magnitudes = spectrum_from_scratch(x[:frame_length])
    print(
        f"{name:>8}: flatness {flatness_from_scratch(magnitudes):.2f}, "
        f"entropy {entropy_from_scratch(magnitudes):.2f}"
    )

### ③ mirpy

`mir.flatness` and `mir.entropy` are MIRtoolbox `mirflatness` and `mirentropy`.

In [ ]:
flat = mir.flatness(a, frame=True)
ent = mir.entropy(a, frame=True)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(flat.frame_centers, flat.data[0, 0], label="flatness")
ax.plot(ent.frame_centers, ent.data[0, 0], label="entropy")
part_lines(ax)
ax.set_xlabel("Time (s)")
ax.set_ylim(0, 1.05)
ax.legend()
plt.show()

### ④ librosa

`librosa.feature.spectral_flatness` works by default on the **power** spectrum (the magnitude squared), which makes small values smaller still. With `power=1.0` it uses the magnitude, as MIRtoolbox does. librosa has no spectral entropy; use the formula in ② on `librosa.stft`.

In [ ]:
librosa_flat_power = librosa.feature.spectral_flatness(S=librosa_spec)[0]
librosa_flat = librosa.feature.spectral_flatness(S=librosa_spec, power=1.0)[0]

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(librosa_times, librosa_flat_power, label="power spectrum (default)")
ax.plot(librosa_times, librosa_flat, label="magnitude spectrum (power=1.0)")
part_lines(ax)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Flatness")
ax.legend()
plt.show()

### ⑤ Essentia

Essentia's `Flatness` uses the magnitude. Its `Entropy` gives the Shannon entropy in bits (base-2 logarithm) of a spectrum scaled to sum to 1; divide by $\log_2 N$ to get MIRtoolbox's 0-to-1 scale.

In [ ]:
def essentia_flatness_entropy(spectra):
    """Flatness and relative entropy of each Essentia spectrum."""
    flatness = es.Flatness()
    entropy = es.Entropy()
    rows = []
    for magnitudes in spectra:
        p = magnitudes / magnitudes.sum()
        rows.append((flatness(magnitudes), entropy(p) / np.log2(p.size)))
    return np.array(rows).T


if es is None:
    print("Essentia is not available; skipped.")
else:
    ess_flat, ess_entropy = essentia_flatness_entropy(essentia_spec)

### ⑥ Side by side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3), sharey=True)
axes[0].plot(flat.frame_centers, flat.data[0, 0], linewidth=3, alpha=0.5, label="mirpy")
axes[0].plot(librosa_times, librosa_flat, label="librosa (power=1.0)")
axes[1].plot(ent.frame_centers, ent.data[0, 0], linewidth=3, alpha=0.5, label="mirpy")
if es is not None:
    axes[0].plot(ess_times, ess_flat, "--", label="Essentia")
    axes[1].plot(ess_times, ess_entropy, "--", label="Essentia")
axes[0].set_title("Flatness")
axes[1].set_title("Entropy")
for ax in axes:
    part_lines(ax)
    ax.set_xlabel("Time (s)")
    ax.legend()
plt.show()

- All three rank the sounds the same way: sine, then sawtooth, then noise.
- **librosa** uses the power spectrum by default; pass `power=1.0` to work on the magnitude like the others.
- Only **mirpy** has the 0-to-1 entropy built in; with **Essentia**, divide by $\log_2 N$.

### Exercise

Mix the 440 Hz sine with more and more noise: noise levels 0, 0.01, 0.03, 0.1 and 0.3. Plot the average flatness and entropy against the noise level. Which of the two reacts first to a little noise?

In [ ]:
# Your answer here.

Solution (click to show):

In [ ]:
levels = [0, 0.01, 0.03, 0.1, 0.3]
flatness_values, entropy_values = [], []
for level in levels:
    mix = mir.audio(sine + level * rng.standard_normal(t.size), sr=sr)
    flatness_values.append(np.nanmean(mir.flatness(mix, frame=True).data))
    entropy_values.append(np.nanmean(mir.entropy(mix, frame=True).data))

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(levels, flatness_values, marker="o", label="flatness")
ax.plot(levels, entropy_values, marker="o", label="entropy")
ax.set_xlabel("Noise level")
ax.legend()
plt.show()

## 5. Roll-off and brightness

### ① Idea

Two ways to measure the amount of high frequencies:

- the **roll-off** is the frequency below which a fixed share of the spectrum lies. MIRtoolbox uses 85 %, following Tzanetakis and Cook (2002); others have proposed 95 % (Pohle, Pampalk and Widmer, 2005);
- the **brightness** turns the question around: it fixes a frequency, the cut-off, and measures the share of the spectrum above it (Juslin, 2000). MIRtoolbox's default cut-off is 1500 Hz; 1000 Hz (Laukka, Juslin and Bresin, 2005) and 3000 Hz (Juslin, 2000) have also been proposed.

### ② From scratch

Add up the spectrum from low to high frequencies (a **cumulative sum**) and divide by the total. The roll-off is where this curve reaches 0.85; the brightness is 1 minus its value at the cut-off.

In [ ]:
magnitudes = spectrum_from_scratch(saw[:frame_length])
cumulative = np.cumsum(magnitudes) / magnitudes.sum()
rolloff_freq = freqs[np.argmax(cumulative >= 0.85)]
brightness_value = magnitudes[freqs > 1500].sum() / magnitudes.sum()

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(freqs, cumulative)
ax.axhline(0.85, color="C3", linestyle="--")
ax.axvline(rolloff_freq, color="C3", label=f"roll-off: {rolloff_freq:.0f} Hz")
ax.axvspan(
    1500, sr / 2, color="C1", alpha=0.15, label=f"above 1500 Hz: brightness {brightness_value:.2f}"
)
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Share of the spectrum below")
ax.set_title("Sawtooth")
ax.legend(loc="lower right")
plt.show()

### ③ mirpy

`mir.rolloff` and `mir.brightness` are MIRtoolbox `mirrolloff` and `mirbrightness`. The options are `threshold` (MATLAB `'Threshold'`) and `cutoff` (MATLAB `'CutOff'`).

In [ ]:
roll = mir.rolloff(a, frame=True)
bright = mir.brightness(a, frame=True)

fig, axes = plt.subplots(2, 1, figsize=(8, 4.5), sharex=True)
axes[0].plot(roll.frame_centers, roll.data[0, 0])
axes[0].set_ylabel("Roll-off (Hz)")
axes[1].plot(bright.frame_centers, bright.data[0, 0])
axes[1].set_ylabel("Brightness")
for ax in axes:
    part_lines(ax)
axes[-1].set_xlabel("Time (s)")
plt.show()

### ④ librosa

`librosa.feature.spectral_rolloff` uses 85 % by default (`roll_percent`). librosa has no brightness function, but it is one line on the spectrogram.

In [ ]:
librosa_roll = librosa.feature.spectral_rolloff(S=librosa_spec, sr=sr, n_fft=n_fft)[0]
librosa_bright = librosa_spec[librosa_freqs > 1500].sum(axis=0) / librosa_spec.sum(axis=0)

### ⑤ Essentia

Essentia's `RollOff` and `EnergyBandRatio` (the share of energy in a band, here from 1500 Hz to the top) do the same jobs. Both work on the **energy**, the magnitude squared. Squaring makes the strong low harmonics count even more, so Essentia's roll-off is lower and its brightness smaller for the sawtooth.

In [ ]:
def essentia_rolloff_brightness(spectra):
    """Roll-off (Hz) and share of energy above 1500 Hz of each Essentia spectrum."""
    rolloff = es.RollOff(sampleRate=sr)
    band_ratio = es.EnergyBandRatio(sampleRate=sr, startFrequency=1500, stopFrequency=sr / 2)
    return np.array([(rolloff(m), band_ratio(m)) for m in spectra]).T


if es is None:
    print("Essentia is not available; skipped.")
else:
    ess_roll, ess_bright = essentia_rolloff_brightness(essentia_spec)

### ⑥ Side by side

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
axes[0].plot(roll.frame_centers, roll.data[0, 0], linewidth=3, alpha=0.5, label="mirpy")
axes[0].plot(librosa_times, librosa_roll, label="librosa")
axes[1].plot(bright.frame_centers, bright.data[0, 0], linewidth=3, alpha=0.5, label="mirpy")
axes[1].plot(librosa_times, librosa_bright, label="librosa (one line)")
if es is not None:
    axes[0].plot(ess_times, ess_roll, "--", label="Essentia (energy)")
    axes[1].plot(ess_times, ess_bright, "--", label="Essentia (energy)")
axes[0].set_ylabel("Roll-off (Hz)")
axes[1].set_ylabel("Brightness")
for ax in axes:
    part_lines(ax)
    ax.legend(loc="upper left")
axes[-1].set_xlabel("Time (s)")
plt.show()

- **mirpy** and **librosa** add up the magnitudes, and their curves match.
- **Essentia** adds up the energy (magnitude squared). For the sawtooth, whose low harmonics are strongest, this gives a lower roll-off and a smaller brightness. For the sine and the noise the results are close.
- When you report a roll-off or brightness, say which threshold or cut-off you used, and whether it was computed on magnitude or energy.

### Exercise

Compute the brightness of the trumpet phrase with the three cut-offs proposed in the literature: 1000, 1500 and 3000 Hz. Plot the three curves together. Which cut-off separates the notes best?

In [ ]:
# Your answer here.

Solution (click to show):

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
for cutoff in (1000, 1500, 3000):
    curve = mir.brightness(trumpet, frame=True, cutoff=cutoff)
    ax.plot(curve.frame_centers, curve.data[0, 0], label=f"{cutoff} Hz")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Brightness")
ax.legend()
plt.show()

A higher cut-off gives smaller values, since less of the spectrum lies above it. There is no single right cut-off; choose one that suits your instruments, and report it.

## Summary

| MIRtoolbox | mirpy | librosa | Essentia |
|---|---|---|---|
| `mirspectrum(a, 'Frame')` | `mir.spectrum(a, frame=True)` | `np.abs(librosa.stft(y))` | `Windowing` + `Spectrum` |
| `mirzerocross(a)` | `mir.zerocross(a)` | `feature.zero_crossing_rate` (both ways, per sample) | `ZeroCrossingRate` (both ways, per sample) |
| `mircentroid(a)` | `mir.centroid(a)` | `feature.spectral_centroid` | `Centroid(range=sr/2)` |
| `mirspread(a)` | `mir.spread(a)` | `feature.spectral_bandwidth` | `CentralMoments` + `DistributionShape` |
| `mirskewness(a)` | `mir.skewness(a)` | from `stft` | `DistributionShape` |
| `mirkurtosis(a)` | `mir.kurtosis(a)` | from `stft` | `DistributionShape` (minus 3) |
| `mirflatness(a)` | `mir.flatness(a)` | `feature.spectral_flatness(power=1.0)` | `Flatness` |
| `mirentropy(a)` | `mir.entropy(a)` | from `stft` | `Entropy` ÷ log₂ N |
| `mirrolloff(a)` | `mir.rolloff(a)` | `feature.spectral_rolloff` | `RollOff` (energy) |
| `mirbrightness(a)` | `mir.brightness(a)` | from `stft` | `EnergyBandRatio` (energy) |

## Next

C4b describes the spectrum with MFCCs, follows its changes over time with spectral flux, and turns all these curves into a table for statistics.

## References

As cited in the MIRtoolbox 1.8.1 User's Manual.

- Lartillot, O., Toiviainen, P. "A Matlab Toolbox for Musical Feature Extraction From Audio", International Conference on Digital Audio Effects, Bordeaux, 2007. *MIRtoolbox.*
- Juslin, P. N. (2000). Cue utilization in communication of emotion in music performance: relating performance to perception. Journal of Experimental Psychology: Human Perception and Performance, 26(6), 1797-813. *Brightness; 3000 Hz cut-off.*
- R. Koch, A Tutorial in Probability and Statistics, http://web.cecs.pdx.edu/~roy/tutorial/Stat_int.htm. *Explanations of spread and skewness.*
- Laukka, P., Juslin, P. N., and Bresin, R. (2005). A dimensional approach to vocal expression of emotion. Cognition and Emotion, 19, 633-653. *1000 Hz brightness cut-off.*
- Pohle, T., E. Pampalk and G. Widmer (2005). Evaluation of Frequently Used Audio Features for Classification of Music Into Perceptual Categories. Proceedings of the Fourth International Workshop on Content-Based Multimedia Indexing (CBMI'05), Riga, Latvia, June 21-23. *95 % roll-off.*
- Shannon, C. E. 1948. "A Mathematical Theory of Communication." Bell Systems Technical Journal 27:379–423. *Entropy.*
- Tzanetakis, Cook. Musical genre classification of audio signals. IEEE Tr. Speech and Audio Processing, 10(5), 293-302, 2002. *85 % roll-off.*